# 🎤 Voice Assistant Starter

**AI Learning Playground — Educational Quickstart Blueprint**

Build, run, and deploy a voice assistant that combines:
- **Whisper Large V3 Turbo** (transformers pipeline) for speech-to-text transcription
- **Llama 3.1 8B Instruct GGUF** (LlamaCpp) for natural-language responses
- **XTTS v2** (CoquiTTS) for text-to-speech synthesis of the response

> ⚠️ **Prerequisite:** Run **[project-setup.ipynb](project-setup.ipynb)** first to install
> dependencies, validate your GPU, authenticate with Hugging Face, and download all models.

---

## What This Notebook Covers

| Step | Topic | Key Concept |
|------|-------|-------------|
| 1 | Configure Settings | Loading `voice.yaml`, resolving model paths |
| 2 | Initialize Model | Instantiating `VoiceModel` with LlamaCpp + Whisper + XTTS |
| 3 | Demo (audio) | WAV/MP3 file → Whisper → LLM → XTTS voice response |
| 4 | GPU Monitoring | VRAM usage during inference |
| 5 | Register Model | Logging to MLflow as `AIStudio-EQ-Voice` |
| 6 | Verify | Loading registered model, full audio round-trip |

## Voice Pipeline Architecture

```
┌─────────────────────────┐
│  Input (audio_base64)   │  ← base64-encoded WAV/MP3/OGG from Streamlit UI
└──────────┬──────────────┘
           │
           ▼
   ┌───────────────────────────────┐
   │  Whisper STT (transformers)    │  ← /home/jovyan/local/whisper-large-v3-turbo/
   └───────────────┬───────────────┘
                   │  transcription text
                   ▼
   ┌───────────────────────────────┐
   │  LLM — Llama 3.1 8B (GGUF)   │
   └───────────────┬───────────────┘
                   │  response text
                   ▼
   ┌───────────────────────────────┐
   │  XTTS v2 TTS (CoquiTTS)      │  ← /home/jovyan/local/xtts-v2/  (bundled as MLflow artifact)
   └───────────────┬───────────────┘
                   │
                   ▼
   ┌─────────────────────────────────────────────────────────┐
   │  outputs: answer (text), messages (JSON), response_audio │
   └─────────────────────────────────────────────────────────┘
```


<div class="alert alert-block alert-warning">
<b>⚠️ Responsible AI Notice — Guardrails Not Included:</b> This notebook builds a three-stage voice pipeline: <b>Whisper STT → Llama 3.1 LLM → XTTS v2 TTS</b>. Each stage introduces unique risks — transcription errors from audio, LLM hallucinations, and harmful content being synthesised directly into audio output. Review outputs at each stage before deploying to users.
<br><br>
<b>💡 Tip:</b> This notebook does not include guardrails. To add safety across the full voice pipeline, consider these open-source tools:
<ul>
  <li><a href="https://github.com/NVIDIA/NeMo-Guardrails"><b>NeMo Guardrails</b></a> — Apply conversational rails at the LLM stage to keep responses on-topic and policy-compliant before passing text to XTTS.</li>
  <li><a href="https://github.com/protectai/llm-guard"><b>LLM Guard</b></a> — Scan post-transcription text for jailbreak attempts, and screen LLM output for harmful content before audio synthesis.</li>
  <li><a href="https://github.com/guardrails-ai/guardrails"><b>Guardrails AI</b></a> — Validate and constrain LLM text responses before they are passed to the XTTS synthesiser.</li>
  <li><a href="https://huggingface.co/meta-llama/Llama-Guard-3-8B"><b>Llama Guard</b></a> — Classify post-transcription user turns and LLM responses as safe or unsafe before audio is synthesised.</li>
</ul>
</div>

In [1]:
import os
import sys
import time
import logging
import warnings
import plotly.io as pio

sys.path.insert(0, "..")

# ── Warning & logging suppression ─────────────────────────────────────────────
# These lines hide informational noise (deprecation notices, version-mismatch
# alerts) so the notebook output stays clean and focused on what you're learning.
# As a learning exercise, try commenting them out one at a time to see what
# each library would normally print during model loading and inference.
os.environ["MLFLOW_LOGGING_LEVEL"] = "ERROR"  # must be set before `import mlflow`
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

pio.renderers.default = "plotly_mimetype+notebook"

start_time = time.time()
print("⏱️  Notebook started")

⏱️  Notebook started


## 2. Configure Settings

Load `configs/voice.yaml` with `capability: voice`.
Note the **two** model paths: `model_path` for the LLM and `stt_model_path` for Whisper.

In [2]:
import os
from src.utils import load_config

config = load_config("../configs/voice.yaml")

model_path     = os.environ.get("MODEL_ARTIFACTS_PATH", config.get("model_path", ""))
stt_model_path = config.get("stt_model_path", "")
tts_model_path = config.get("tts_model_path", "")
context_window = config.get("context_window", 8192)

print(f"Capability     : {config.get('capability')}")
print(f"LLM path       : {model_path}")
print(f"Whisper path   : {stt_model_path}")
print(f"TTS path       : {tts_model_path}")
print(f"Context window : {context_window} tokens")


Capability     : voice
LLM path       : /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
Whisper path   : /home/jovyan/local/whisper-large-v3-turbo
TTS path       : /home/jovyan/local/xtts-v2
Context window : 8192 tokens


## 3. Verify Assets

In [3]:
from src.utils import log_asset_status

# tts_model_path is now a directory — verify the key model file inside it.
tts_config_path = os.path.join(tts_model_path, "config.json") if tts_model_path else ""

assets = [
    {"name": "LLM (GGUF)",            "path": model_path,                              "required": True},
    {"name": "Whisper STT (dir)",      "path": stt_model_path,                          "required": False},
    {"name": "XTTS v2 dir (TTS)",      "path": tts_model_path,                          "required": False},
    {"name": "XTTS v2 config.json",    "path": tts_config_path,                         "required": False},
    {"name": "Config YAML",            "path": "../configs/voice.yaml",                  "required": True},
    {"name": "Sample audio",           "path": "../data/input/sample_question.mp3",      "required": False},
    {"name": "Voice demo UI",          "path": "../demo/voice/streamlit/main.py",        "required": False},
]

log_asset_status(assets)

## 4. Initialize VoiceModel

The `VoiceModel` loads the LLM at construction time.
Whisper is loaded lazily on the first audio request — saving VRAM until needed.

In [4]:
from src.mlflow.models.voice import VoiceModel
from src.gpu_monitor import GPUMonitor

print("Initializing VoiceModel...")

model = VoiceModel(
    config=config,
    model_path=model_path,
)
monitor = GPUMonitor()

print("\n✅ VoiceModel ready")
print(f"   LLM loaded    : {'yes' if model.llm is not None else 'no (check model_path)'}")
print(f"   Whisper status: loads on first audio request")


Initializing VoiceModel...


llama_context: n_ctx_per_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



✅ VoiceModel ready
   LLM loaded    : yes
   Whisper status: loads on first audio request


## 5. Demo: Audio Mode

The full three-stage pipeline:
1. Decode the base64 audio and save to a temporary WAV file
2. Transcribe with **Whisper Large V3 Turbo** via `AutoProcessor` + `AutoModelForSpeechSeq2Seq`
3. Pass the transcription to **Llama 3.1 8B** for a response
4. Synthesise the response as speech with **XTTS v2** (CoquiTTS) — result in `response_audio`


In [5]:
import base64
import os
import pandas as pd
import IPython.display as ipd

sample_audio_path = "../data/input/sample_question.mp3"

if os.path.exists(sample_audio_path):
    with open(sample_audio_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode("utf-8")

    result = model.predict(pd.DataFrame([{
        "question":     "",
        "audio_base64": audio_b64,
    }]))

    print("Audio mode result:")
    print("─" * 60)
    print(result["answer"].iloc[0])

    # ── TTS playback ─────────────────────────────────────────────────────────
    response_audio_b64 = result["response_audio"].iloc[0]
    if response_audio_b64:
        audio_bytes = base64.b64decode(response_audio_b64)
        print("\n🔊 Playing synthesised response (XTTS v2)...")
        display(ipd.Audio(audio_bytes, rate=24000, autoplay=True))
    else:
        print("\nℹ️  TTS output unavailable (XTTS model not loaded or synthesis skipped).")
else:
    print("ℹ️  No sample audio found at:", sample_audio_path)
    print("   Place any MP3/WAV file there to test audio mode.")


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Audio mode result:
────────────────────────────────────────────────────────────
Transcription: Write me a two-sentence poem about a robot that just learned how to hear.

Response: Here is a two-sentence poem about a robot that just learned how to hear:

With newfound ears, the robot stands,
Listening to the world, with a mechanical hand.

🔊 Playing synthesised response (XTTS v2)...


## 7. GPU Monitoring

In [6]:
monitor.display_dashboard()


## 8. Register with MLflow

Register as **`AIStudio-EQ-Voice`** — fully independent from the other three models.
The `capability: voice` key in the config routes serving requests to `VoiceModel`.

In [7]:
import mlflow

EXPERIMENT_NAME = 'VoiceAssistant-StarterExperiment'

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name= EXPERIMENT_NAME)

ARTIFACT_PATH = "AIStudio-EQ-Voice"
MODEL_NAME    = "AIStudio-EQ-Voice"

print(f"Artifact path  : {ARTIFACT_PATH}")
print(f"Registered as  : {MODEL_NAME}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")

Artifact path  : AIStudio-EQ-Voice
Registered as  : AIStudio-EQ-Voice
Tracking URI   : /phoenix/mlflow


In [8]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# VoiceModel input schema:
#   question      — kept for schema backward-compatibility; not used for inference
#   audio_base64  — base64-encoded audio bytes (WAV, MP3, OGG, FLAC); required
input_schema = Schema([
    ColSpec("string", "question"),     # Unused; present for schema compat
    ColSpec("string", "audio_base64"), # Base64-encoded audio for Whisper transcription
])

output_schema = Schema([
    ColSpec("string", "answer"),         # Transcription + LLM response text
    ColSpec("string", "messages"),       # JSON conversation history
    ColSpec("string", "response_audio"), # Base64-encoded WAV (XTTS v2 TTS; empty if TTS disabled)
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print("Input  : question (string, unused), audio_base64 (string, required)")
print("Output : answer (string), messages (JSON string), response_audio (base64 WAV string)")


Input  : question (string, unused), audio_base64 (string, required)
Output : answer (string), messages (JSON string), response_audio (base64 WAV string)


In [9]:
from src.mlflow.logger import Logger

with mlflow.start_run(run_name=f"register-{ARTIFACT_PATH}") as run:
    Logger.log_model(
        signature      = signature,
        artifact_path  = ARTIFACT_PATH,
        config_path    = "../configs/voice.yaml",
        # All three model assets are bundled as artifacts so the serving
        # container never needs to access /home/jovyan/local/ at runtime.
        # Logger copies files as-is and directories as named sub-folders under
        # models/ (e.g. xtts-v2/ → models/xtts-v2/).  Loader resolves the
        # paths back from models/ when constructing VoiceModel at serve time.
        model_paths    = {
            "model_path":     model_path,      # LLM GGUF file
            "stt_model_path": stt_model_path,  # Whisper model directory
            "tts_model_path": tts_model_path,  # XTTS v2 directory → bundled as models/xtts-v2/
        },
        demo_folder    = "../demo/voice",
    )
    run_id = run.info.run_id

print(f"✅ Model logged | Run ID: {run_id}")
print(f"   LLM      : {os.path.basename(model_path) if model_path else 'n/a'}")
print(f"   Whisper  : {os.path.basename(stt_model_path) if stt_model_path else 'n/a'}")
print(f"   XTTS v2  : {os.path.basename(tts_model_path) if tts_model_path else 'n/a (TTS disabled)'}")

model_uri = f"runs:/{run_id}/{ARTIFACT_PATH}"
reg = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"✅ Registered  : {MODEL_NAME} v{reg.version}")


✅ Model logged | Run ID: 0ca1315bd5c6400fae5245ba66b4b575
   LLM      : Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
   Whisper  : whisper-large-v3-turbo
   XTTS v2  : xtts-v2


Registered model 'AIStudio-EQ-Voice' already exists. Creating a new version of this model...


✅ Registered  : AIStudio-EQ-Voice v38


Created version '38' of model 'AIStudio-EQ-Voice'.


## 9. Verify Registration

In [10]:
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

# Verify with audio — text mode is not supported
if os.path.exists(sample_audio_path):
    with open(sample_audio_path, "rb") as f:
        verify_b64 = base64.b64encode(f.read()).decode("utf-8")

    test_result = loaded_model.predict(pd.DataFrame([{
        "question":     "",
        "audio_base64": verify_b64,
    }]))

    print("✅ Loaded model response:")
    print(test_result["answer"].iloc[0][:300])
else:
    print("⚠️  sample_question.mp3 not found — skipping audio verification")
    test_result = None

llama_context: n_ctx_per_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ Loaded model response:
Transcription: Write me a two-sentence poem about a robot that just learned how to hear.

Response: Here is a two-sentence poem about a robot that just learned how to hear:

With newfound ears, the robot stands,
Listening to the world, with a mechanical hand.


In [11]:

# ── TTS playback verification ─────────────────────────────────────────────────
# Pull the base64-encoded WAV from the registered model's prediction and
# play it inline so we can hear the full voice pipeline end-to-end.

import base64
import IPython.display as ipd

if test_result is not None:
    response_audio_b64 = test_result["response_audio"].iloc[0]

    if response_audio_b64:
        audio_bytes = base64.b64decode(response_audio_b64)
        print("🔊 Playing XTTS v2 synthesised response...")
        display(ipd.Audio(audio_bytes, rate=24000, autoplay=True))
    else:
        print("ℹ️  response_audio is empty — XTTS model was not loaded during this run.")
        print("   Ensure tts_model_path is set and the XTTS v2 directory was logged as an artifact.")
else:
    print("ℹ️  No test_result — sample audio was not available for verification.")


🔊 Playing XTTS v2 synthesised response...


In [12]:
elapsed = time.time() - start_time
print(f"⏱️  Total notebook time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

⏱️  Total notebook time: 228.7s (3.8 min)


---

## ✅ What We Accomplished

| Step | Result |
|------|--------|
| Environment | CUDA + soundfile verified, dependencies installed |
| VoiceModel | Initialized with LlamaCpp (Whisper loads on demand) |
| Text demo | Multiple questions answered via LLM directly |
| Audio demo | WAV → Whisper → LLM pipeline demonstrated |
| GPU Monitor | VRAM usage during inference visible |
| Registration | `AIStudio-EQ-Voice` registered in Model Registry |
| Verification | Loaded model returned correct answer |

## All 4 Models Registered

| Model Name | Capability | Notebook |
|---|---|---|
| `AIStudio-EQ-Chatbot` | Conversational Q&A | `chatbot-starter.ipynb` |
| `AIStudio-EQ-ImageGen` | Text-to-image | `image-gen-starter.ipynb` |
| `AIStudio-EQ-Document` | Document RAG Q&A | `document-analyzer-starter.ipynb` |
| `AIStudio-EQ-Voice` | Voice assistant | `voice-assistant-starter.ipynb` |

## Next Steps

- **Deploy in AI Studio:** Open the Model Registry → select any registered model → Deploy → choose the matching Streamlit UI
- **Try real audio:** Record a WAV file and convert to base64 to test the Whisper pipeline
- **Extend the voice pipeline:** Add text-to-speech (TTS) for a full voice-in/voice-out experience